### Project Solution - Goal 1

First we should look at what's in the file itself. Just a few records should be enough.

The file we are working with is:

```python
nyc_parking_tickets_extract.csv
```

Our first goal is to create a lazy iterator that returns a named tuple for every valid data row, with the data converted to more appropriate Python types.

In [1]:
file_name = 'nyc_parking_tickets_extract.csv'

In [2]:
with open(file_name, encoding='utf-8-sig') as f:
    for _ in range(10):
        print(next(f))

Summons Number,Plate ID,Registration State,Plate Type,Issue Date,Violation Code,Vehicle Body Type,Vehicle Make,Violation Description

4006478550,VAD7274,VA,PAS,10/5/2016,5,4D,BMW,BUS LANE VIOLATION

4006462396,22834JK,NY,COM,9/30/2016,5,VAN,CHEVR,BUS LANE VIOLATION

4007117810,21791MG,NY,COM,4/10/2017,5,VAN,DODGE,BUS LANE VIOLATION

4006265037,FZX9232,NY,PAS,8/23/2016,5,SUBN,FORD,BUS LANE VIOLATION

4006535600,N203399C,NY,OMT,10/19/2016,5,SUBN,FORD,BUS LANE VIOLATION

4007156700,92163MG,NY,COM,4/13/2017,5,VAN,FRUEH,BUS LANE VIOLATION

4006687989,MIQ600,SC,PAS,11/21/2016,5,VN,HONDA,BUS LANE VIOLATION

4006943052,2AE3984,MD,PAS,2/1/2017,5,SW,LINCO,BUS LANE VIOLATION

4007306795,HLG4926,NY,PAS,5/30/2017,5,SUBN,TOYOT,BUS LANE VIOLATION



So we should notice a few things:

1. The first row contains the column headers.
2. Every row comes from the file as text.
3. The date values need to be converted to `datetime.date` objects.
4. Some numeric-looking columns should be converted to integers.
5. We should not assume the data is perfectly clean.

Also, although this small extract is simple enough that `split(',')` would appear to work, it is better practice to use Python's built-in `csv` module. That way, the solution is more robust if a text field ever contains a comma.

#### Column Definitions and Named Tuple

Let's start by reading the column headers and one sample row from the file.

In [3]:
import csv

with open(file_name, encoding='utf-8-sig', newline='') as f:
    reader = csv.reader(f)
    column_headers = next(reader)
    sample_data = next(reader)

In [4]:
column_headers

['Summons Number',
 'Plate ID',
 'Registration State',
 'Plate Type',
 'Issue Date',
 'Violation Code',
 'Vehicle Body Type',
 'Vehicle Make',
 'Violation Description']

In [5]:
sample_data

['4006478550',
 'VAD7274',
 'VA',
 'PAS',
 '10/5/2016',
 '5',
 '4D',
 'BMW',
 'BUS LANE VIOLATION']

In [6]:
list(zip(column_headers, sample_data))

[('Summons Number', '4006478550'),
 ('Plate ID', 'VAD7274'),
 ('Registration State', 'VA'),
 ('Plate Type', 'PAS'),
 ('Issue Date', '10/5/2016'),
 ('Violation Code', '5'),
 ('Vehicle Body Type', '4D'),
 ('Vehicle Make', 'BMW'),
 ('Violation Description', 'BUS LANE VIOLATION')]

The column names in the file contain spaces and capital letters. For a named tuple, it is more convenient to use lowercase names with underscores.

In [7]:
column_names = [header.strip().replace(' ', '_').lower()
                for header in column_headers]

column_names

['summons_number',
 'plate_id',
 'registration_state',
 'plate_type',
 'issue_date',
 'violation_code',
 'vehicle_body_type',
 'vehicle_make',
 'violation_description']

Based on the sample data and the project description, the data types we want are:

    0. summons_number: integer
    1. plate_id: string
    2. registration_state: string
    3. plate_type: string
    4. issue_date: date
    5. violation_code: integer
    6. vehicle_body_type: string
    7. vehicle_make: string
    8. violation_description: string

Now we can create the named tuple class.

In [8]:
from collections import namedtuple

Ticket = namedtuple('Ticket', column_names)
Ticket

__main__.Ticket

#### Reading Data Rows Lazily

Every time we want to process the actual data, we need to skip the header row. So let's create a small generator function that yields just the data rows.

This function is lazy: it does not read the entire file into memory.

In [9]:
def read_data(file_name=file_name):
    with open(file_name, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        next(reader)  # skip header row
        yield from reader

We can test the generator by asking it for just a few rows.

In [10]:
from itertools import islice

rows = read_data()
for row in islice(rows, 5):
    print(row)

['4006478550', 'VAD7274', 'VA', 'PAS', '10/5/2016', '5', '4D', 'BMW', 'BUS LANE VIOLATION']
['4006462396', '22834JK', 'NY', 'COM', '9/30/2016', '5', 'VAN', 'CHEVR', 'BUS LANE VIOLATION']
['4007117810', '21791MG', 'NY', 'COM', '4/10/2017', '5', 'VAN', 'DODGE', 'BUS LANE VIOLATION']
['4006265037', 'FZX9232', 'NY', 'PAS', '8/23/2016', '5', 'SUBN', 'FORD', 'BUS LANE VIOLATION']
['4006535600', 'N203399C', 'NY', 'OMT', '10/19/2016', '5', 'SUBN', 'FORD', 'BUS LANE VIOLATION']


Notice that each row is now a list of strings. The next step is to convert each string value to the appropriate type.

#### Parser Functions

Let's write small parser functions for integers, dates, and strings.

Each parser will return a default value if the incoming value is missing or invalid. This gives us control over which rows we later keep or reject.

In [11]:
from datetime import datetime


def parse_int(value, *, default=None):
    if value is None:
        return default
    value = str(value).strip()
    if not value:
        return default
    try:
        return int(value)
    except ValueError:
        return default



def parse_date(value, *, default=None):
    if value is None:
        return default
    value = str(value).strip()
    if not value:
        return default
    date_format = '%m/%d/%Y'
    try:
        return datetime.strptime(value, date_format).date()
    except ValueError:
        return default



def parse_string(value, *, default=None):
    if value is None:
        return default
    cleaned = str(value).strip()
    if not cleaned:
        return default
    return cleaned

Let's test these parser functions before we use them in the full solution.

In [12]:
parse_int('123')

123

In [13]:
parse_int('hello', default='N/A')

'N/A'

In [14]:
parse_date('3/28/2018')

datetime.date(2018, 3, 28)

In [15]:
parse_date('31/31/2000', default='N/A')

'N/A'

In [16]:
parse_string('   BMW   ')

'BMW'

In [17]:
parse_string('   ', default='N/A')

'N/A'

#### Column Parsers

Now we need to connect each column with the parser that should be used for that column.

Some fields are required for a useful ticket record, so they will keep the default value of `None` if missing or invalid. Other fields can safely be empty strings, such as the violation description.

In [18]:
from functools import partial

column_parsers = (
    parse_int,                              # summons_number
    parse_string,                           # plate_id
    partial(parse_string, default=''),      # registration_state
    partial(parse_string, default=''),      # plate_type
    parse_date,                             # issue_date
    parse_int,                              # violation_code
    partial(parse_string, default=''),      # vehicle_body_type
    parse_string,                           # vehicle_make
    partial(parse_string, default='')       # violation_description
)

#### Parsing One Row

To parse a row, we can zip together the parsers and the row values. Then we call each parser on its matching field.

We will return a `Ticket` named tuple when the important fields are valid. Otherwise, we will return the specified default value.

In [19]:
def parse_row(row, *, default=None):
    if len(row) != len(column_parsers):
        return default

    parsed_data = [func(field)
                   for func, field in zip(column_parsers, row)]

    if all(item is not None for item in parsed_data):
        return Ticket(*parsed_data)
    else:
        return default

Let's test the row parser on a few rows.

In [20]:
rows = read_data()
for row in islice(rows, 5):
    parsed_row = parse_row(row)
    print(parsed_row)

Ticket(summons_number=4006478550, plate_id='VAD7274', registration_state='VA', plate_type='PAS', issue_date=datetime.date(2016, 10, 5), violation_code=5, vehicle_body_type='4D', vehicle_make='BMW', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006462396, plate_id='22834JK', registration_state='NY', plate_type='COM', issue_date=datetime.date(2016, 9, 30), violation_code=5, vehicle_body_type='VAN', vehicle_make='CHEVR', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4007117810, plate_id='21791MG', registration_state='NY', plate_type='COM', issue_date=datetime.date(2017, 4, 10), violation_code=5, vehicle_body_type='VAN', vehicle_make='DODGE', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4

This gives us the structure we wanted: every valid row is now represented as a named tuple, and fields such as `summons_number`, `issue_date`, and `violation_code` have been converted to better Python data types.

#### Checking for Rows with Missing Required Values

Before building the final iterator, let's check whether any rows fail to parse. This is not part of the lazy pipeline itself; it is just a useful data quality check.

In [22]:
for row_number, row in enumerate(read_data(), start=2):
    parsed_row = parse_row(row)
    if parsed_row is None:
        print('Problem row:', row_number)
        print(list(zip(column_names, row)), end='')

Problem row: 190
[('summons_number', '1413358512'), ('plate_id', '54295PC'), ('registration_state', 'NY'), ('plate_type', 'APP'), ('issue_date', '8/9/2016'), ('violation_code', '19'), ('vehicle_body_type', 'BUS'), ('vehicle_make', ''), ('violation_description', '')]Problem row: 398
[('summons_number', '1418425369'), ('plate_id', 'JYW5248'), ('registration_state', 'PA'), ('plate_type', 'PAS'), ('issue_date', '3/21/2017'), ('violation_code', '21'), ('vehicle_body_type', 'SDN'), ('vehicle_make', ''), ('violation_description', '')]Problem row: 842
[('summons_number', '1406925068'), ('plate_id', '19358JU'), ('registration_state', '99'), ('plate_type', 'COM'), ('issue_date', '8/23/2016'), ('violation_code', '46'), ('vehicle_body_type', 'DELV'), ('vehicle_make', ''), ('violation_description', '')]Problem row: 843
[('summons_number', '8546468965'), ('plate_id', '37489BB'), ('registration_state', 'NY'), ('plate_type', 'OMR'), ('issue_date', '6/12/2017'), ('violation_code', '46'), ('vehicle_body

#### Final Lazy Iterator

Now we can create the final lazy iterator required by Goal 1.

This function reads the CSV row by row, parses each row, and yields only valid `Ticket` named tuples.

In [23]:
def iter_tickets(file_name=file_name):
    for row in read_data(file_name):
        parsed_row = parse_row(row)
        if parsed_row is not None:
            yield parsed_row

Let's test the iterator. Again, we only ask for the first few rows, which means the file is not fully consumed.

In [24]:
tickets = iter_tickets()
for ticket in islice(tickets, 5):
    print(ticket)

Ticket(summons_number=4006478550, plate_id='VAD7274', registration_state='VA', plate_type='PAS', issue_date=datetime.date(2016, 10, 5), violation_code=5, vehicle_body_type='4D', vehicle_make='BMW', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006462396, plate_id='22834JK', registration_state='NY', plate_type='COM', issue_date=datetime.date(2016, 9, 30), violation_code=5, vehicle_body_type='VAN', vehicle_make='CHEVR', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4007117810, plate_id='21791MG', registration_state='NY', plate_type='COM', issue_date=datetime.date(2017, 4, 10), violation_code=5, vehicle_body_type='VAN', vehicle_make='DODGE', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4

We can also inspect one ticket more closely.

In [25]:
ticket = next(iter_tickets())
ticket

Ticket(summons_number=4006478550, plate_id='VAD7274', registration_state='VA', plate_type='PAS', issue_date=datetime.date(2016, 10, 5), violation_code=5, vehicle_body_type='4D', vehicle_make='BMW', violation_description='BUS LANE VIOLATION')

In [26]:
type(ticket.summons_number), type(ticket.issue_date), type(ticket.violation_code)

(int, datetime.date, int)

At this point Goal 1 is complete:

- the file is read lazily;
- each row is converted to a named tuple;
- important fields are parsed into appropriate data types;
- rows with missing required values are skipped.

### Project Solution - Goal 2

Now we need to calculate the number of violations by car make.

For this, a `Counter` is a good fit. Counting itself cannot be completely lazy because we need to keep the running counts in memory. But the input rows are still consumed lazily from `iter_tickets()`.

In [27]:
from collections import Counter


def violation_count_by_make(file_name=file_name):
    return Counter(ticket.vehicle_make
                   for ticket in iter_tickets(file_name))

In [28]:
make_counts = violation_count_by_make()
make_counts

Counter({'TOYOT': 112,
         'HONDA': 106,
         'FORD': 104,
         'CHEVR': 76,
         'NISSA': 70,
         'DODGE': 45,
         'FRUEH': 44,
         'ME/BE': 38,
         'GMC': 35,
         'HYUND': 35,
         'BMW': 34,
         'LEXUS': 26,
         'INTER': 25,
         'JEEP': 22,
         'NS/OT': 18,
         'SUBAR': 18,
         'INFIN': 13,
         'LINCO': 12,
         'CHRYS': 12,
         'ACURA': 12,
         'AUDI': 12,
         'VOLVO': 12,
         'MITSU': 11,
         'ISUZU': 10,
         'CADIL': 9,
         'KIA': 8,
         'VOLKS': 8,
         'HIN': 6,
         'KENWO': 5,
         'ROVER': 5,
         'BUICK': 5,
         'MAZDA': 5,
         'MERCU': 4,
         'JAGUA': 3,
         'SMART': 3,
         'PORSC': 3,
         'WORKH': 2,
         'SATUR': 2,
         'SCION': 2,
         'SAAB': 2,
         'HINO': 2,
         'FIR': 1,
         'OLDSM': 1,
         'PETER': 1,
         'CITRO': 1,
         'GEO': 1,
         'YAMAH': 1,
   

The `most_common` method gives us the results sorted from highest count to lowest count.

In [29]:
make_counts.most_common()

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35),
 ('BMW', 34),
 ('LEXUS', 26),
 ('INTER', 25),
 ('JEEP', 22),
 ('NS/OT', 18),
 ('SUBAR', 18),
 ('INFIN', 13),
 ('LINCO', 12),
 ('CHRYS', 12),
 ('ACURA', 12),
 ('AUDI', 12),
 ('VOLVO', 12),
 ('MITSU', 11),
 ('ISUZU', 10),
 ('CADIL', 9),
 ('KIA', 8),
 ('VOLKS', 8),
 ('HIN', 6),
 ('KENWO', 5),
 ('ROVER', 5),
 ('BUICK', 5),
 ('MAZDA', 5),
 ('MERCU', 4),
 ('JAGUA', 3),
 ('SMART', 3),
 ('PORSC', 3),
 ('WORKH', 2),
 ('SATUR', 2),
 ('SCION', 2),
 ('SAAB', 2),
 ('HINO', 2),
 ('FIR', 1),
 ('OLDSM', 1),
 ('PETER', 1),
 ('CITRO', 1),
 ('GEO', 1),
 ('YAMAH', 1),
 ('BSA', 1),
 ('MINI', 1),
 ('PONTI', 1),
 ('SPRI', 1),
 ('PLYMO', 1),
 ('UPS', 1),
 ('FIAT', 1),
 ('UD', 1),
 ('UTILI', 1),
 ('GMCQ', 1),
 ('STAR', 1),
 ('AM/T', 1),
 ('MI/F', 1)]

We can print the result in a cleaner format as well.

In [30]:
for make, count in make_counts.most_common():
    print(f'{make:>10}: {count}')

     TOYOT: 112
     HONDA: 106
      FORD: 104
     CHEVR: 76
     NISSA: 70
     DODGE: 45
     FRUEH: 44
     ME/BE: 38
       GMC: 35
     HYUND: 35
       BMW: 34
     LEXUS: 26
     INTER: 25
      JEEP: 22
     NS/OT: 18
     SUBAR: 18
     INFIN: 13
     LINCO: 12
     CHRYS: 12
     ACURA: 12
      AUDI: 12
     VOLVO: 12
     MITSU: 11
     ISUZU: 10
     CADIL: 9
       KIA: 8
     VOLKS: 8
       HIN: 6
     KENWO: 5
     ROVER: 5
     BUICK: 5
     MAZDA: 5
     MERCU: 4
     JAGUA: 3
     SMART: 3
     PORSC: 3
     WORKH: 2
     SATUR: 2
     SCION: 2
      SAAB: 2
      HINO: 2
       FIR: 1
     OLDSM: 1
     PETER: 1
     CITRO: 1
       GEO: 1
     YAMAH: 1
       BSA: 1
      MINI: 1
     PONTI: 1
      SPRI: 1
     PLYMO: 1
       UPS: 1
      FIAT: 1
        UD: 1
     UTILI: 1
      GMCQ: 1
      STAR: 1
      AM/T: 1
      MI/F: 1


#### A Slightly More General Counting Function

Since we now have clean named tuples, we can easily count by any field, not just vehicle make. This is a useful bit of added functionality while still keeping the solution simple.

In [31]:
def count_by(field_name, file_name=file_name):
    if field_name not in Ticket._fields:
        raise ValueError(f'{field_name!r} is not a valid field name.')

    return Counter(getattr(ticket, field_name)
                   for ticket in iter_tickets(file_name))

For example, here are the counts by registration state.

In [32]:
count_by('registration_state').most_common()

[('NY', 777),
 ('NJ', 94),
 ('PA', 29),
 ('FL', 14),
 ('VA', 10),
 ('CT', 10),
 ('IN', 7),
 ('MA', 6),
 ('TX', 5),
 ('TN', 4),
 ('SC', 3),
 ('MD', 3),
 ('AZ', 3),
 ('CA', 3),
 ('RI', 3),
 ('AL', 3),
 ('GA', 3),
 ('NC', 2),
 ('IA', 2),
 ('ME', 2),
 ('MN', 2),
 ('99', 2),
 ('OH', 1),
 ('ON', 1),
 ('WY', 1),
 ('DE', 1),
 ('IL', 1),
 ('MI', 1),
 ('OR', 1),
 ('VT', 1)]

And here are the counts by violation code.

In [33]:
count_by('violation_code').most_common()

[(21, 161),
 (36, 140),
 (38, 121),
 (14, 86),
 (46, 51),
 (37, 49),
 (40, 47),
 (20, 46),
 (71, 45),
 (7, 40),
 (19, 25),
 (47, 18),
 (70, 14),
 (16, 13),
 (31, 13),
 (24, 12),
 (69, 12),
 (74, 12),
 (50, 11),
 (5, 10),
 (48, 7),
 (84, 7),
 (42, 6),
 (53, 6),
 (10, 5),
 (9, 3),
 (17, 3),
 (51, 3),
 (68, 3),
 (77, 3),
 (78, 3),
 (82, 3),
 (98, 3),
 (13, 2),
 (85, 2),
 (18, 1),
 (23, 1),
 (26, 1),
 (27, 1),
 (35, 1),
 (52, 1),
 (64, 1),
 (75, 1),
 (80, 1),
 (91, 1)]

#### Optional Lazy Filters

Because `iter_tickets()` yields one ticket at a time, we can build small lazy filters on top of it.

In [34]:
def tickets_by_make(make, file_name=file_name):
    make = parse_string(make, default='')
    make = make.upper()
    for ticket in iter_tickets(file_name):
        if ticket.vehicle_make.upper() == make:
            yield ticket



def tickets_by_state(state, file_name=file_name):
    state = parse_string(state, default='')
    state = state.upper()
    for ticket in iter_tickets(file_name):
        if ticket.registration_state.upper() == state:
            yield ticket

Let's look at a few Ford tickets.

In [35]:
for ticket in islice(tickets_by_make('FORD'), 5):
    print(ticket)

Ticket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006535600, plate_id='N203399C', registration_state='NY', plate_type='OMT', issue_date=datetime.date(2016, 10, 19), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=5093620865, plate_id='AD80228', registration_state='AZ', plate_type='PAS', issue_date=datetime.date(2016, 9, 24), violation_code=7, vehicle_body_type='TK', vehicle_make='FORD', violation_description='FAILURE TO STOP AT RED LIGHT')
Ticket(summons_number=5092733548, plate_id='EVX4041', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 7, 26), violation_code=7, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='FAILURE TO STOP AT RED LIGHT')


And a few tickets from New York registrations.

In [36]:
for ticket in islice(tickets_by_state('NY'), 5):
    print(ticket)

Ticket(summons_number=4006462396, plate_id='22834JK', registration_state='NY', plate_type='COM', issue_date=datetime.date(2016, 9, 30), violation_code=5, vehicle_body_type='VAN', vehicle_make='CHEVR', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4007117810, plate_id='21791MG', registration_state='NY', plate_type='COM', issue_date=datetime.date(2017, 4, 10), violation_code=5, vehicle_body_type='VAN', vehicle_make='DODGE', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_number=4006535600, plate_id='N203399C', registration_state='NY', plate_type='OMT', issue_date=datetime.date(2016, 10, 19), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE VIOLATION')
Ticket(summons_num

#### Final Answer

The main answer to Goal 2 is the `make_counts` object, sorted below.

In [37]:
final_result = violation_count_by_make()
final_result.most_common()

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35),
 ('BMW', 34),
 ('LEXUS', 26),
 ('INTER', 25),
 ('JEEP', 22),
 ('NS/OT', 18),
 ('SUBAR', 18),
 ('INFIN', 13),
 ('LINCO', 12),
 ('CHRYS', 12),
 ('ACURA', 12),
 ('AUDI', 12),
 ('VOLVO', 12),
 ('MITSU', 11),
 ('ISUZU', 10),
 ('CADIL', 9),
 ('KIA', 8),
 ('VOLKS', 8),
 ('HIN', 6),
 ('KENWO', 5),
 ('ROVER', 5),
 ('BUICK', 5),
 ('MAZDA', 5),
 ('MERCU', 4),
 ('JAGUA', 3),
 ('SMART', 3),
 ('PORSC', 3),
 ('WORKH', 2),
 ('SATUR', 2),
 ('SCION', 2),
 ('SAAB', 2),
 ('HINO', 2),
 ('FIR', 1),
 ('OLDSM', 1),
 ('PETER', 1),
 ('CITRO', 1),
 ('GEO', 1),
 ('YAMAH', 1),
 ('BSA', 1),
 ('MINI', 1),
 ('PONTI', 1),
 ('SPRI', 1),
 ('PLYMO', 1),
 ('UPS', 1),
 ('FIAT', 1),
 ('UD', 1),
 ('UTILI', 1),
 ('GMCQ', 1),
 ('STAR', 1),
 ('AM/T', 1),
 ('MI/F', 1)]

#### Complete Reusable Solution

Finally, here is a small wrapper function that runs the main project solution and returns the violations by make.

In [38]:
def solve(file_name=file_name):
    return violation_count_by_make(file_name)


solve().most_common(10)

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35)]

### Notes on Lazy Evaluation

- `read_data()` is lazy: it yields one CSV row at a time.
- `iter_tickets()` is lazy: it yields one parsed named tuple at a time.
- `tickets_by_make()` and `tickets_by_state()` are lazy filters.
- `Counter` is necessarily eager because it must store the counts.
- We avoided materializing the whole data set except when intentionally displaying small samples with `islice`.